# 03 Features 
Builds time on ball, first touch, pitch zone, and outcome columns for each reception. 

In [20]:
import pandas as pd
ev = pd.read_pickle("data/nwsl_2023_events.pkl")
ev = ev.sort_values(["match_id", "index"]).reset_index(drop=True)
ev["t"] = pd.to_timedelta(ev["timestamp"]).dt.total_seconds()

reps = pd.read_pickle("data/receptions.pkl")
reps.shape

(112874, 15)

In [21]:
cols = ["index", "t", "type", "player", "possession"]
ev[cols].iloc[100:115]

,index,t,type,player,possession
100,101,75.054,Pass,Emily Ann Fox,2
101,102,76.385,Ball Receipt*,Tyler Lussi,2
102,103,76.385,Carry,Tyler Lussi,2
103,104,76.795,Pass,Tyler Lussi,2
104,105,77.194,Pressure,Lo’eau LaBonta,2
105,106,77.249,Ball Receipt*,Denise O'Sullivan,2
106,107,77.249,Pass,Denise O'Sullivan,2
107,108,78.456,Ball Receipt*,Tyler Lussi,2
108,109,78.456,Carry,Tyler Lussi,2
109,110,80.953,Pass,Tyler Lussi,2


In [22]:
release_types = ["Pass", "Shot", "Dispossessed", "Miscontrol", "Clearance"]

rel = ev[ev["type"].isin(release_types)]
rel = rel[["match_id", "index", "t", "player", "possession", "type"]]
rel.shape

(141261, 6)

In [23]:
r = reps[["match_id", "index", "t", "player", "possession"]].assign(type="Ball Receipt*")

seq = pd.concat([r,rel])
seq = seq.sort_values(["match_id", "index"]).reset_index(drop=True)
seq.head(10)

,match_id,index,t,player,possession,type
0,3881483,5,1.693,Kerolin Nicoli Israel Ferraz,2,Pass
1,3881483,6,3.422,Meredith Speck,2,Ball Receipt*
2,3881483,8,4.190,Meredith Speck,2,Pass
3,3881483,9,5.450,Kiara Shaelene Pickett,2,Ball Receipt*
4,3881483,11,6.261,Kiara Shaelene Pickett,2,Pass
5,3881483,12,8.525,Ryan Williams,2,Ball Receipt*
6,3881483,14,9.390,Ryan Williams,2,Pass
7,3881483,15,11.125,Kaleigh Kurtz,2,Ball Receipt*
8,3881483,17,11.903,Kaleigh Kurtz,2,Pass
9,3881483,18,13.114,Denise O'Sullivan,2,Ball Receipt*


In [24]:
nxt = seq.groupby("match_id").shift(-1)
nxt.head(5)

,index,t,player,possession,type
0,6.0,3.422,Meredith Speck,2.0,Ball Receipt*
1,8.0,4.190,Meredith Speck,2.0,Pass
2,9.0,5.450,Kiara Shaelene Pickett,2.0,Ball Receipt*
3,11.0,6.261,Kiara Shaelene Pickett,2.0,Pass
4,12.0,8.525,Ryan Williams,2.0,Ball Receipt*


In [25]:
same = (
    (nxt["player"] ==seq["player"]) &
    (nxt["possession"] == seq["possession"]) &
    (nxt["type"] != "Ball Receipt*")
)
same.head(5)

0    False
1     True
2    False
3     True
4    False
dtype: bool

In [26]:
seq["time_on_ball"] = (nxt["t"] - seq["t"]).where(same) 
seq["release_type"] = nxt["type"].where(same)
seq.head(5)

,match_id,index,t,player,possession,type,time_on_ball,release_type
0,3881483,5,1.693,Kerolin Nicoli Israel Ferraz,2,Pass,NaN,NaN
1,3881483,6,3.422,Meredith Speck,2,Ball Receipt*,0.768,Pass
2,3881483,8,4.190,Meredith Speck,2,Pass,NaN,NaN
3,3881483,9,5.450,Kiara Shaelene Pickett,2,Ball Receipt*,0.811,Pass
4,3881483,11,6.261,Kiara Shaelene Pickett,2,Pass,NaN,NaN


In [27]:
tob = seq[seq["type"] == "Ball Receipt*"] [["match_id", "index", "time_on_ball", "release_type"]]

reps = reps.merge(tob, on=["match_id", "index"], how ="left")
reps.shape

(112874, 17)

In [28]:
print(reps["time_on_ball"].notna().mean())
reps["time_on_ball"].describe()

0.7575526693481227


count    85508.000000
mean         1.577884
std          1.576000
min          0.000000
25%          0.395000
50%          1.216000
75%          2.188000
max         24.080000
Name: time_on_ball, dtype: float64

In [29]:
reps.sort_values("time_on_ball", ascending=False).head(5)

,id,match_id,index,period,t,minute,team,player,position,location,under_pressure,possession,ball_receipt_outcome,received_ok,role,time_on_ball,release_type
60196,a64ea6f4-f257-43bf-84a3-584066364ac1,3881555,2984,2,2443.511,85,Portland Thorns,Sophia Olivia Wilson,Center Forward,"[62.7, 61.8]",False,168,NaN,True,ST,24.080,Dispossessed
11891,7769afe9-0da6-4bd0-9327-1e570a1cb09e,3881497,3095,2,3295.583,99,NJ/NY Gotham FC,Yazmeen Ryan,Right Wing,"[89.8, 10.4]",False,216,NaN,True,W,20.727,Pass
22990,400d8275-a3e8-4766-b44e-77f6d65d67fd,3881510,3400,2,2576.472,87,Orlando Pride,Messiah Bright,Center Forward,"[41.7, 8.3]",True,182,NaN,True,ST,16.982,Dispossessed
69711,25d07d9f-bf2e-40b9-8374-bbfd0bdee36d,3881567,2843,2,2625.379,88,Houston Dash,Maria Guadalupe Sanchez Morales,Left Wing Back,"[56.8, 9.0]",False,192,NaN,True,FB,16.307,Pass
85800,22f87ba2-05a5-4929-abb1-3bf2b9e21ac1,3881587,2163,2,890.480,59,Chicago Red Stars,Tatumn Milazzo,Right Back,"[31.2, 75.2]",False,115,NaN,True,FB,15.471,Pass


In [30]:
# 24 second outlier check 
cols =  ["player", "role", "minute", "time_on_ball", "release_type", "under_pressure"]
reps.sort_values("time_on_ball", ascending=False)[cols].head(10)

,player,role,minute,time_on_ball,release_type,under_pressure
60196,Sophia Olivia Wilson,ST,85,24.080,Dispossessed,False
11891,Yazmeen Ryan,W,99,20.727,Pass,False
22990,Messiah Bright,ST,87,16.982,Dispossessed,True
69711,Maria Guadalupe Sanchez Morales,FB,88,16.307,Pass,False
85800,Tatumn Milazzo,FB,59,15.471,Pass,False
48118,Messiah Bright,ST,32,15.322,Pass,False
99476,Messiah Bright,ST,22,15.306,Pass,False
38104,Michelle Alozie,W,28,15.155,Pass,True
92204,Kaleigh Kurtz,CB,25,15.126,Pass,False
14968,Caitlin Cosme,CB,40,14.760,Pass,False


## Part 2: Good First Touch

In [36]:
TOUCH_WINDOW = 2.0

bad_touch =(
    (reps["release_type"] == "Miscontrol") |
    ((reps["release_type"] == "Dispossessed") & (reps["time_on_ball"] <= TOUCH_WINDOW))
)

reps["good_touch"] = reps["received_ok"] & reps["time_on_ball"].notna() & ~bad_touch
reps["good_touch"].mean()

np.float64(0.7034392331272038)

In [37]:
judged = reps["received_ok"] & reps["time_on_ball"].notna()

print(judged.sum())
reps.loc[judged, "good_touch"].mean()

85137


np.float64(0.9326144919365258)

In [38]:
matches = pd.read_pickle("data/matches.pkl")
matches[matches["match_date"] == "2023-11-12"][["match_id", "match_date", "home_team", "away_team"]]

,match_id,match_date,home_team,away_team
10,3915998,2023-11-12,Seattle Reign,NJ/NY Gotham FC


In [39]:
FINAL_ID = 3915998

j = reps[judged & (reps["match_id"] == FINAL_ID)]
print(len(j), (~j["good_touch"]).sum())

good = j[j["good_touch"]].sample(10, random_state=42)
bad = j[~j["good_touch"]].sample(10, random_state=42)

sample = pd.concat([good, bad]).sample(frac=1, random_state=42)
sample.shape

609 44


(20, 18)

## Part 3 : Pitch Zones 

In [42]:
reps = reps.drop(columns="thirds", errors="ignore")

reps["x"] = reps["location"].str[0]
reps["y"] = reps["location"].str[1]

reps["third"] = pd.cut(
    reps["x"],
    bins=[0, 40, 80, 120],
    labels=["defensive", "middle", "final"],
    include_lowest=True
)

reps["central"] = reps["y"].between(18, 62)

reps["third"].value_counts()

third
middle       51417
final        34547
defensive    26910
Name: count, dtype: int64

In [43]:
reps.groupby("third")["time_on_ball"].median()

/var/folders/6t/7rsnpsl57gs140xc9md0hw1r0000gn/T/ipykernel_50168/2312352428.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  reps.groupby("third")["time_on_ball"].median()


third
defensive    1.3395
middle       1.2000
final        1.0930
Name: time_on_ball, dtype: float64

## Part 4: Outcomes 

In [45]:
shots = ev[ev["type"] == "Shot"][["match_id","possession", "index", "team", "shot_statsbomb_xg"]]
shots = shots.rename(columns={"index": "shot_index", "team": "shot_team"})

pairs = reps[["id","match_id","possession", "index", "team"]].merge(
    shots, on=["match_id","possession"]
)
pairs = pairs[(pairs["shot_index"] > pairs["index"]) & (pairs["shot_team"] == pairs["team"])]

xg_after = pairs.groupby("id")["shot_statsbomb_xg"].sum()
reps["xg_after"] = reps["id"].map(xg_after).fillna(0)

print((reps["xg_after"] > 0).mean())
reps["xg_after"].describe()

0.12525470879033257


count    112874.000000
mean          0.013833
std           0.060078
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.174838
Name: xg_after, dtype: float64

In [ ]:
WINDOW = 10

later = ev[ev["type"].isin(["Shot", "Pass"])][
    ["match_id", "possession","index","t","team", "type","shot_statsbomb_xg"]
    ].rename(columns={"index": "ev_index", "t": "ev_t", "team": "ev_team"})